# Redrob Candidate Discovery & Ranking Pipeline Demo
=====================================================

This notebook demonstrates the Intelligent Candidate Discovery & Ranking Pipeline for the founding **Senior AI Engineer** role at Redrob AI.

### Core Pipeline Mechanics:
1. **Phase 1: Universal Elimination (Honeypot Filter)**: Enforces 7 structural integrity checks in `honeypot.py` (e.g. expert skill duration limits, overlapping full-time dates, impossible degree timelines, and tool release date checks) to weed out synthetic profiles prior to scoring.
2. **Phase 2: Dual-Vector Semantic Matching**: Encodes target job descriptions into positive, negative, and skill vectors, matching them semantically against duration-weighted candidate career text via local `bge-small-en-v1.5` embeddings.
3. **Phase 3: Soft YOE Scaling & Grounding Checks**: Scales outlier YOE scores softly to remain compliant with job description guidelines. Skills verification checks are performed to ensure declared skills are semantically grounded in career descriptions.
4. **Phase 4: Multi-Dimensional Behavioral Signal Modifier**: Weights candidate profiles using a clamped composite multiplier built from 20 distinct platform indicators (activity recency, response rates, assessment scores, notice periods, company type, education, relocation, and career trajectory).
5. **Phase 5: Grounded LLM Brief Generation**: Generates factual hiring recommendation briefs using a local `Qwen 2.5 1.5B GGUF` model, enforced by technology-grounding post-processing filters to prevent LLM hallucinations.

### Google Colab Environment Setup (Automatic)
If running in Google Colab, we clone the Git repository, navigate into the directory, and install the required dependencies automatically. If running locally, this step is skipped.

In [1]:
# Check if running in Google Colab
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    print("Running in Google Colab. Setting up workspace...")
    # 1. Clone the repository
    !git clone https://github.com/Karan825/RetrievAl.git

    # 2. Change directory into the repository
    %cd RetrievAl

    # 3. Install llama-cpp-python from precompiled CPU wheels (avoids 10+ min build time)
    !pip install llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cpu

    # 4. Install remaining required packages
    !pip install -r requirements.txt
else:
    print("Running in local workspace. Skipping Colab setup.")

Running in Google Colab. Setting up workspace...
Cloning into 'RetrievAl'...
remote: Enumerating objects: 208, done.
remote: Counting objects: 100% (42/42), done.
remote: Compressing objects: 100% (15/15), done.
remote: Total 208 (delta 27), reused 30 (delta 27), pack-reused 166 (from 1)
Receiving objects: 100% (208/208), 6.42 MiB | 21.49 MiB/s, done.
Resolving deltas: 100% (105/105), done.
/content/RetrievAl
Looking in indexes: https://pypi.org/simple, https://abetlen.github.io/llama-cpp-python/whl/cpu
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22.7/22.7 MB 48.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 61.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 103.7 MB/s eta 0:00:00


### Step 1: System Check
We verify the existence of `sample_candidates.json` and the pre-computed JD embeddings/metadata files.

In [2]:
import os
import sys
from pathlib import Path

# Paths to verify
sample_candidates = Path("sample_candidates.json")
jd_embeddings = Path("vrd/jd_embeddings.npz")
jd_metadata = Path("vrd/jd_metadata.json")

print("Checking required files:")
print(f"  sample_candidates.json : {'EXISTS' if sample_candidates.exists() else 'MISSING'}")
print(f"  vrd/jd_embeddings.npz  : {'EXISTS' if jd_embeddings.exists() else 'MISSING'}")
print(f"  vrd/jd_metadata.json   : {'EXISTS' if jd_metadata.exists() else 'MISSING'}")

if not (sample_candidates.exists() and jd_embeddings.exists() and jd_metadata.exists()):
    raise FileNotFoundError("Required files are missing! Please check the workspace setup.")

Checking required files:
  sample_candidates.json : EXISTS
  vrd/jd_embeddings.npz  : EXISTS
  vrd/jd_metadata.json   : EXISTS


### Step 1.5: Optional Candidate File Upload
If you want to test the ranking pipeline on a custom candidate dataset (<=100 profiles), run this cell to upload it. Supported formats: JSON array (`.json`) or JSON Lines (`.jsonl`).

If you want to use the pre-loaded `sample_candidates.json` dataset, simply skip this cell or select no file (cancel) to use the pre-loaded data.

In [3]:
# Option to upload your own custom candidates file
try:
    from google.colab import files
    print("Prompting for custom candidate file upload (JSON or JSONL format). Click cancel or close the prompt to use the pre-loaded dataset:")
    uploaded = files.upload()
    if uploaded:
        custom_file = list(uploaded.keys())[0]
        print(f"Uploaded custom candidates file: {custom_file}")
    else:
        custom_file = None
        print("No file uploaded. Using pre-loaded sample_candidates.json dataset.")
except Exception as e:
    custom_file = None
    print("Not running in Google Colab environment or using pre-loaded dataset. Defaulting to sample_candidates.json.")

Prompting for custom candidate file upload (JSON or JSONL format). Click cancel or close the prompt to use the pre-loaded dataset:


No file uploaded. Using pre-loaded sample_candidates.json dataset.


### Step 2: Convert Candidate File to JSONL Format
The main ranking pipeline expects candidate data in the JSONL format (one candidate JSON object per line). We convert the dataset (whether pre-loaded or uploaded custom file) into a temporary `temp_candidates.jsonl` file.

In [4]:
import json
import os
import shutil

# Determine input file path
input_file = "sample_candidates.json"
try:
    if 'custom_file' in globals() and custom_file and os.path.exists(custom_file):
        input_file = custom_file
except NameError:
    pass

print(f"Using input candidates dataset: {input_file}")

if input_file.endswith(".jsonl"):
    print("Input file is already in JSONL format. Copying to temp_candidates.jsonl...")
    shutil.copy(input_file, "temp_candidates.jsonl")
else:
    print("Converting JSON array to JSONL format...")
    with open(input_file, "r", encoding="utf-8") as f:
        candidates = json.load(f)

    with open("temp_candidates.jsonl", "w", encoding="utf-8") as f:
        for c in candidates:
            f.write(json.dumps(c) + "\n")
    print(f"Success! Converted {len(candidates)} candidates and saved to temp_candidates.jsonl.")

Using input candidates dataset: sample_candidates.json
Converting JSON array to JSONL format...
Success! Converted 50 candidates and saved to temp_candidates.jsonl.


### Step 3: Run the Ranking Pipeline
We run the ranker CLI (`rank.py`) with `temp_candidates.jsonl` as input and output the results to `submission.csv`.

In [5]:
# Execute the ranking wrapper script
!python rank.py --candidates temp_candidates.jsonl --out submission.csv

[rank.py] Launching main ranker...
[main_ranker] Loading JD Brain from /content/RetrievAl/vrd/jd_embeddings.npz...
Fetching 14 files:   0% 0/14 [00:00<?, ?it/s]Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
Fetching 14 files: 100% 14/14 [00:02<00:00,  4.72it/s]
Download complete: 100% 401M/401M [00:02<00:00, 136MB/s]                BGE model download complete!

qwen2.5-1.5b-instruct-q4_k_m.gguf:   0% 0.00/1.12G [00:00<?, ?B/s]
qwen2.5-1.5b-instruct-q4_k_m.gguf:   0% 0.00/1.12G [00:00<?, ?B/s]
qwen2.5-1.5b-instruct-q4_k_m.gguf:   0% 0.00/1.12G [00:00<?, ?B/s]
qwen2.5-1.5b-instruct-q4_k_m.gguf:   0% 0.00/1.12G [00:00<?, ?B/s]
qwen2.5-1.5b-instruct-q4_k_m.gguf:   0% 0.00/1.12G [00:00<?, ?B/s]
qwen2.5-1.5b-instruct-q4_k_m.gguf:   0% 0.00/1.12G [00:00<?, ?B/s]
qwen2.5-1.5b-instruct-q4_k_m.gguf:   0% 0.00/1.12G [00:00<?, ?B/s]
qwen2.5-1.5b-instruct-q4_k_m.gguf:   0% 0.00/1.12G [00:00<?, ?B/s]
qwen2.5-1

### Step 4: Display the Ranked Output
We read the generated `submission.csv` and display the top ranked candidates in a clean Pandas DataFrame.

In [6]:
import pandas as pd

# Load the generated submission file
df = pd.read_csv("submission.csv")

# Set display options for clean reading
pd.set_option("display.max_colwidth", None)

print(f"Total Ranked Candidates: {len(df)}")
df.head(10)

Total Ranked Candidates: 3


,candidate_id,rank,score,reasoning
0,CAND_0000031,1,95.7453,"Recommendation Systems Engineer - 6.0 YOE with Swiggy. Skilled in Pinecone, Embeddings, Information Retrieval. Aligned with must-haves."
1,CAND_0000032,2,13.9224,"Skilled in Embeddings, Python, OpenCV over 8.1 YOE as .NET Developer at Cognizant. Excellent fit for retrieval requirements; concern: consulting background (Cognizant, HCL), 150-day notice period."
2,CAND_0000038,3,12.0307,"Java Developer with 6.7 YOE with Swiggy. Focused on Weaviate, GraphQL, Azure. Strong match - strongly aligned with search engineering needs. Note: low response rate (20%), 90-day notice period."


### Step 5: Cleanup Temporary Files
We clean up the temporary JSONL file created during Step 2.

In [7]:
if os.path.exists("temp_candidates.jsonl"):
    os.remove("temp_candidates.jsonl")
    print("Temporary file temp_candidates.jsonl cleaned up.")

Temporary file temp_candidates.jsonl cleaned up.
